# Week 09 — Feature Engineering & Hyperparameter Optimization
### Maximizing Performance: Baseline vs Optimized Model
**Next Gen Software Hub Pvt. Ltd. — AI & ML Internship**


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif

df = pd.read_csv("datasets/telecom_churn_features.csv")
print(f"Loaded {df.shape[0]} records.")
df.head()


## 1. Feature Engineering
Create domain interaction features:


In [ ]:
# Ratio of dropped calls per month of tenure
df["Dropped_per_Month"] = df["Calls_Dropped"] / (df["Tenure_Months"] + 1)

# Average spend per tenure month
df["Avg_Spend_Ratio"] = df["Total_Charges"] / (df["Tenure_Months"] * df["Monthly_Charges"] + 1e-5)

# High risk interaction flag
df["Frustration_Score"] = df["Calls_Dropped"] * df["Customer_Support_Tickets"]
df.head()


## 2. Baseline Model Performance


In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train, y_train)
baseline_preds = baseline_rf.predict(X_test)
baseline_auc = roc_auc_score(y_test, baseline_rf.predict_proba(X_test)[:, 1])

print(f"Baseline Random Forest ROC-AUC: {baseline_auc:.4f}")
print(classification_report(y_test, baseline_preds))


## 3. Systematic Hyperparameter Tuning with GridSearchCV


In [ ]:
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [3, 5, 8, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
best_preds = best_rf.predict(X_test)
best_auc = roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1])

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Tuned Random Forest ROC-AUC: {best_auc:.4f} (Baseline: {baseline_auc:.4f})")
print(f"Net Improvement: +{(best_auc - baseline_auc)*100:.2f}%")
